# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [ ]:
import carla
import time
import random
import cv2
import queue
import threading
import json
import math
import numpy as np
import paho.mqtt.client as mqtt
from datetime import datetime

# Initialize client and connect to the CARLA server daemon
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()
blueprint_library = world.get_blueprint_library()

In [ ]:
def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

---
## 1. Infrastructure Sensing & World Transformation

### Mathematical Foundation: 2D Image Space to 3D World Space
To send meaningful target tracking coordinates to an Ego vehicle, a static roadside infrastructure camera must map a detected object's pixel coordinate $(u, v)$ back into a 3D World coordinate $(X_w, Y_w, Z_w)$.

This is achieved using the camera Intrinsic Matrix ($K$) and Extrinsic Matrix ($[R|t]$):

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

Given that the ground plane can be structurally approximated ($Z_w \approx 0$), we solve for the scaling factor $\lambda$ to project inverse coordinates from the pixel plane back through the inverted rigid transformation matrix of the camera mount location.

### Configurable Parameters Guide
- **Camera Extrinsics**: Positioning height ($z \ge 6.0\text{m}$) and dramatic downward pitch ($\text{pitch} \approx -35^\circ$) are vital to eliminate self-occlusion artifacts within blind intersections.
- **Intrinsic Matrix Fields**: Tuning Resolution ($W, H$) and Horizontal Field of View ($\text{FOV}$) determines pixel density per meter at long range.

In [ ]:
def get_camera_intrinsic_matrix(width, height, fov):
    """Computes the intrinsic matrix K for a pinhole camera model."""
    focal = width / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = focal
    K[1, 1] = focal
    K[0, 2] = width / 2.0
    K[1, 2] = height / 2.0
    return K

def spawn_roadside_camera(world, transform, width=800, height=600, fov=90, tick=0.05):
    """Spawns a static infrastructure sensor frame."""
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    return world.spawn_actor(bp, transform)

def image_to_bgr(image):
    """Converts CARLA raw image array to standard BGR layout."""
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()

---
## 2. V2X MQTT Server Architecture & Message Serialization

### Asynchronous I2V V2X Message Design
Real-world infrastructure messaging relies on standardized frameworks. In this implementation, the infrastructure packages objects using an asymmetric JSON schema over an external or emulated MQTT message broker channel.

#### V2X Data Payload Spec (JSON String)
```json
{
  "timestamp": 1716885368.123,
  "station_id": 9901,
  "detected_objects": [
    {
      "class": "pedestrian",
      "world_x": 124.52,
      "world_y": -45.12,
      "world_z": 0.05,
      "confidence": 0.94
    }
  ]
}
```

In [ ]:
class V2XMessageBroker:
    """
    Hybrid V2X Network Gateway. Attempts a real MQTT connection to an external 
    server process. Safe-falls back to a simulated local broker if no daemon is found.
    """
    def __init__(self, broker_address="localhost", port=1883, topic="v2x/intersection/5", latency_ms=40):
        self.topic = topic
        self.latest_payload = None
        self.is_live_mqtt = False
        
        # Local fallback fields
        self.fallback_queue = queue.Queue()
        self.latency_seconds = latency_ms / 1000.0
        
        try:
            self.client = mqtt.Client()
            self.client.on_message = self._on_message_callback
            self.client.connect(broker_address, port, keepalive=2)
            self.client.loop_start()
            self.client.subscribe(self.topic)
            self.is_live_mqtt = True
            print(f"[V2X SERVER] Connected successfully to external MQTT Server on {broker_address}:{port}")
        except Exception:
            print("[V2X SERVER] No external MQTT daemon detected. Launching local Thread-Safe Network Server Emulator.")
            
    def _on_message_callback(self, client, userdata, msg):
        """Asynchronous callback parsing live network server data incoming frames."""
        try:
            self.latest_payload = json.loads(msg.payload.decode('utf-8'))
        except Exception:
            pass

    def publish(self, payload_dict):
        """Broadcasts payload package across the selected server channel."""
        if self.is_live_mqtt:
            self.client.publish(self.topic, json.dumps(payload_dict))
        else:
            release_time = time.time() + self.latency_seconds
            self.fallback_queue.put((release_time, json.dumps(payload_dict)))

    def receive_latest(self):
        """Polls the absolute freshest message packet cleared by the broker layer."""
        if self.is_live_mqtt:
            return self.latest_payload
            
        now = time.time()
        latest_valid_msg = None
        temp_list = []
        
        while not self.fallback_queue.empty():
            try:
                item = self.fallback_queue.get_nowait()
                if now >= item[0]:
                    latest_valid_msg = item[1]
                else:
                    temp_list.append(item)
            except queue.Empty:
                break
                
        for pending_item in temp_list:
            self.fallback_queue.put(pending_item)
            
        if latest_valid_msg:
            return json.loads(latest_valid_msg)
        return None

    def disconnect(self):
        """Cleanly drops connections during script termination."""
        if self.is_live_mqtt:
            self.client.loop_stop()
            self.client.disconnect()
            print("[V2X SERVER] Connection to live server closed.")

---
## 3. End-to-End System Integration & Occlusion Scenarios

### Local Perception vs Shared Infrastructure Fusion
- **Without V2X Assistance**: The Ego vehicle relies solely on its front bumper line-of-sight sensors, resulting in emergency braking and a high collision rate.
- **With V2X Assistance**: The Ego vehicle continuously consumes the message broker queue, projects the shared target location into its local coordinate system, and computes early proactive speed reduction profiles.

In [ ]:
def infrastructure_processing_loop(cam_sensor, broker, stop_event, camera_transform):
    """
    Simulates the infrastructure node. Ingests raw camera frames, isolates target
    actors through simulated 2D detection, and issues V2X telemetry to the server.
    """
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            bgr_frame = image_to_bgr(image)
            
            world_context = cam_sensor.get_world()
            actors = world_context.get_actors()
            pedestrians = actors.filter("walker.pedestrian.*")
            cyclists = actors.filter("vehicle.bpc.bicycle")
            vulnerable_road_users = list(pedestrians) + list(cyclists)
            
            detected_payloads = []
            
            for vru in vulnerable_road_users:
                vru_loc = vru.get_transform().location
                if cam_sensor.get_transform().location.distance(vru_loc) < 45.0:
                    detected_payloads.append({
                        "class": "pedestrian" if "walker" in vru.type_id else "cyclist",
                        "world_x": vru_loc.x,
                        "world_y": vru_loc.y,
                        "world_z": vru_loc.z,
                        "confidence": round(random.uniform(0.92, 0.99), 2)
                    })
            
            if detected_payloads:
                msg = {
                    "timestamp": time.time(),
                    "station_id": 5501,
                    "detected_objects": detected_payloads
                }
                broker.publish(msg)
                
            cv2.putText(bgr_frame, f"V2X SERVER NODE - Tracking: {len(detected_payloads)} VRUs", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.imshow("Infrastructure Node Monitoring Feed", bgr_frame)
            cv2.waitKey(1)
            
        except queue.Empty:
            continue
            
    cam_sensor.stop()
    cv2.destroyWindow("Infrastructure Node Monitoring Feed")

In [ ]:
# System Operational Configuration Flags
USE_V2X_ASSISTANCE = True  # Set to False to evaluate the unassisted occlusion control baseline
SCENARIO_DURATION = 14.0   # Operational timeout limit per run

# Define Actor Storage arrays
spawned_actors = []
network_stop_signal = threading.Event()
network_stop_signal.clear()

try:
    # 1. Setup Environment Maps and Spawning Points
    intersection_center = carla.Location(x=150.0, y=130.0, z=0.5)
    
    # Spawn Ego Vehicle
    ego_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
    ego_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=1.0), carla.Rotation(yaw=0.0))
    ego_vehicle = world.spawn_actor(ego_bp, ego_tf)
    spawned_actors.append(ego_vehicle)
    
    # Spawn Vulnerable Road User (Crossing Pedestrian behind the blind wall)
    ped_bp = random.choice(blueprint_library.filter("walker.pedestrian.*"))
    ped_tf = carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0))
    ped_actor = world.spawn_actor(ped_bp, ped_tf)
    spawned_actors.append(ped_actor)
    
    # Spawn Infrastructure Camera overlooking the intersection from high vantage point
    infra_tf = carla.Transform(carla.Location(x=155.0, y=125.0, z=9.5), carla.Rotation(pitch=-35.0, yaw=110.0))
    infra_camera = spawn_roadside_camera(world, infra_tf)
    spawned_actors.append(infra_camera)
    
    # 2. Instantiate V2X System Broker Client Integration
    v2x_server_node = V2XMessageBroker(broker_address="localhost", port=1883, latency_ms=40)
    
    infra_thread = threading.Thread(
        target=infrastructure_processing_loop, 
        args=(infra_camera, v2x_server_node, network_stop_signal, infra_tf),
        daemon=True
    )
    infra_thread.start()
    
    # Initialize Metrics Logging Arrays
    min_distance_to_target = float("inf")
    v2x_lead_time_discovered = None
    collision_detected = False
    
    # Stabilize client sync ticks
    world.tick()
    time.sleep(0.5)
    
    # Start Moving the Pedestrian into the blind intersection crossing path
    ped_control = carla.WalkerControl(direction=carla.Vector3D(0, -1, 0), speed=1.8)
    ped_actor.apply_control(ped_control)
    
    start_time = time.time()
    print(f">> Executing Occlusion Simulation Run. V2X Assistance Status: {USE_V2X_ASSISTANCE}")
    
    while time.time() - start_time < SCENARIO_DURATION:
        world.tick()
        
        # Update Editor Spectator behind ego tracking array
        move_spectator_to(ego_vehicle.get_transform(), spectator, distance=14.0, z=5.0, pitch=-20.0)
        
        # Calculate Real-Time Spatial Distance profiles
        ego_loc = ego_vehicle.get_transform().location
        ped_loc = ped_actor.get_transform().location
        current_distance = ego_loc.distance(ped_loc)
        
        if current_distance < min_distance_to_target:
            min_distance_to_target = current_distance
            
        if current_distance < 1.9:
            collision_detected = True
            
        # Extract vehicle telemetry speed data
        v_vector = ego_vehicle.get_velocity()
        current_speed_kmh = 3.6 * math.sqrt(v_vector.x**2 + v_vector.y**2 + v_vector.z**2)
        
        # Default Control Profile (Unassisted Cruise Control Baseline)
        target_throttle = 0.45
        target_brake = 0.0
        
        # 3. Process Server Ingested V2X Message Payloads
        incoming_broadcast = v2x_server_node.receive_latest()
        
        if USE_V2X_ASSISTANCE and incoming_broadcast is not None:
            for obj in incoming_broadcast["detected_objects"]:
                target_pos = carla.Location(x=obj["world_x"], y=obj["world_y"], z=obj["world_z"])
                
                # Check alignment constraints relative to ego path tracking
                longitudinal_hazard_dist = target_pos.x - ego_loc.x
                lateral_deviation = abs(target_pos.y - ego_loc.y)
                
                if 0.0 < longitudinal_hazard_dist < 35.0 and lateral_deviation < 3.5:
                    if v2x_lead_time_discovered is None:
                        v2x_lead_time_discovered = time.time() - start_time
                        print(f"   [V2X ALERT] Infrastructure Message Ingested! Lead Time: {v2x_lead_time_discovered:.2f}s")
                    
                    # Proactive safety speed policy deceleration rules
                    if longitudinal_hazard_dist > 16.0:
                        target_throttle = 0.05
                        target_brake = 0.30
                        world.debug.draw_string(ego_loc + carla.Location(z=2.5), "V2X: COOPERATIVE SLOWING", life_time=0.05, color=carla.Color(255, 165, 0))
                    else:
                        target_throttle = 0.0
                        target_brake = 1.0
                        world.debug.draw_string(ego_loc + carla.Location(z=2.5), "V2X: INFRASTRUCTURE CONTROL STOP", life_time=0.05, color=carla.Color(255, 0, 0))
                        
        # Local Sensor Line-of-Sight Emulation (Blind Baseline Fallback Perception)
        if (ped_loc.y - ego_loc.y) < 4.0 and abs(ped_loc.x - ego_loc.x) < 12.0:
            if not USE_V2X_ASSISTANCE:
                world.debug.draw_string(ego_loc + carla.Location(z=2.5), "LOCAL LOS EMERGENCY BRAKE", life_time=0.05, color=carla.Color(255, 0, 0))
                target_throttle = 0.0
                target_brake = 1.0
                
        ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake), steer=0.0))
        world.debug.draw_string(ped_loc + carla.Location(z=2.0), "CROSSING HAZARD", life_time=0.05, color=carla.Color(0, 255, 255))
        
        if int(time.time() - start_time) % 2 == 0 and (time.time() - start_time) % 1.0 < 0.05:
            print(f"   Time={time.time()-start_time:.1f}s | Speed={current_speed_kmh:.1f} km/h | Target Dist={current_distance:.1f}m")
            
        time.sleep(0.03)

    print("\n" + "="*50 + "\n VAL DELIVERABLES: SCENARIO PERFORMANCE METRICS\n" + "="*50)
    print(f" - Avoided Near Misses / Collision Occurred : {'💥 COLLISION DETECTED' if collision_detected else '✅ SUCCESSFUL AVOIDANCE'}")
    print(f" - Minimum Absolute Distance Recorded       : {min_distance_to_target:.2f} meters")
    print(f" - V2X Warning Lead Time Ingestion           : {f'{v2x_lead_time_discovered:.2f} seconds' if v2x_lead_time_discovered else 'N/A (No V2X)'}")
    print("="*50)

finally:
    print(">> Stopping infrastructure processing threads and closing server connections...")
    network_stop_signal.set()
    infra_thread.join(timeout=2.0)
    v2x_server_node.disconnect()
    safe_destroy(spawned_actors)
    print(">> Environment clean up complete.")